# Import Libraries

In [2]:
import pandas as pd
import numpy as np
import spacy
import os
from tqdm import tqdm 
from sklearn.metrics import classification_report
import os
from openai import OpenAI
import time
import random

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_colwidth', None)  # Show full content in each cell
pd.set_option('display.width', 1000)  # Set max width

# Load spaCy's English model
nlp = spacy.load('en_core_web_sm')

# Pre-Processing

In [3]:
label_mapper = {
    'knowledge' : 0,
    'comprehension' : 1,
    'application' : 2,
    'analysis' : 3,
    'synthesis' : 4,
    'evaluation' : 5
}

mapping = {
    'knowledge': 'knowledge',
    'remember': 'knowledge',
    'comprehension': 'comprehension',
    'understand': 'comprehension',
    'application': 'application',
    'apply': 'application',
    'analysis': 'analysis',
    'analyse': 'analysis',
    'evaluation': 'evaluation',
    'evaluate': 'evaluation',
    'synthesis': 'synthesis',
    'create': 'synthesis'
}

q_df = pd.read_csv(os.getcwd().replace('notebook' , 'dataset') + '/dataset4.csv')
queries = q_df['question']
q_df['label'] = q_df['label'].str.lower()
q_df['label'] = q_df['label'].replace(mapping)
label = q_df['label'].str.lower().map(label_mapper)
print(q_df['label'].value_counts())

label
synthesis        29
knowledge        22
evaluation       21
comprehension    20
analysis         19
application      15
Name: count, dtype: int64


# API Setup

In [7]:
# Groq

api_key = os.environ.get("GROQ_API_KEY")

if api_key:
    print('successful')

groq_client = OpenAI(
    base_url = "https://api.groq.com/openai/v1",
    api_key = api_key
)

successful


# Zero-Shot

## GPT-OSS-120B

### Assign Labels

In [8]:
zso_pred_labels = []

for query in tqdm(queries):
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""Given the query below, classify which label it belongs.
                    Labels: [knowledge, comprehension, application, analysis, synthesis, evaluation]
                
                query : {query}""",
            }
        ],
        model="openai/gpt-oss-120b",
    )

    reply = chat_completion.choices[0].message.content.lower()

    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        chat_completion = groq_client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": f"""Extract answer from previous reponse only in one word without punctuation from: 
                        [knowledge , comprehension , application , analysis, synthesis , evaluation]
                    previous response : {reply}""",
                }
            ],
            model="openai/gpt-oss-20b",
        )

        reply = chat_completion.choices[0].message.content.lower()

    zso_pred_labels.append(reply.lower())

100%|██████████| 126/126 [04:51<00:00,  2.31s/it]


In [9]:
print(classification_report(label , [label_mapper[key.lower()] for key in zso_pred_labels]))

              precision    recall  f1-score   support

           0       0.88      0.95      0.91        22
           1       0.72      0.65      0.68        20
           2       0.46      0.40      0.43        15
           3       0.81      0.68      0.74        19
           4       0.68      0.90      0.78        29
           5       0.94      0.76      0.84        21

    accuracy                           0.75       126
   macro avg       0.75      0.72      0.73       126
weighted avg       0.76      0.75      0.75       126



## LLAMA4-Scout

In [10]:
zsl_pred_labels = []

for query in tqdm(queries):
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""Given the query below, classify which label it belongs.
                    Labels: [knowledge, comprehension, application, analysis, synthesis, evaluation]
                
                query : {query}""",
            }
        ],
        model="meta-llama/llama-4-scout-17b-16e-instruct",
    )

    reply = chat_completion.choices[0].message.content.lower()

    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        chat_completion = groq_client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": f"""Extract answer from previous reponse only in one word without punctuation from: 
                        [knowledge , comprehension , application , analysis, synthesis , evaluation]
                    previous response : {reply}""",
                }
            ],
            model="meta-llama/llama-4-scout-17b-16e-instruct",
        )

        reply = chat_completion.choices[0].message.content.lower()

    zsl_pred_labels.append(reply.lower())

100%|██████████| 126/126 [09:35<00:00,  4.57s/it]


In [11]:
print(classification_report(label , [label_mapper[key.lower()] for key in zsl_pred_labels]))

              precision    recall  f1-score   support

           0       0.89      0.73      0.80        22
           1       0.59      0.65      0.62        20
           2       0.45      0.60      0.51        15
           3       0.76      0.68      0.72        19
           4       0.77      0.83      0.80        29
           5       0.89      0.76      0.82        21

    accuracy                           0.72       126
   macro avg       0.73      0.71      0.71       126
weighted avg       0.74      0.72      0.73       126



## INSTRUCTION PROMPT

## GPT-OSS-120B

In [12]:
ipo_pred_labels = []

for query in tqdm(queries):
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""Given the query below, classify which label it belongs.
                    Labels: [knowledge, comprehension, application, analysis, synthesis, evaluation]

                    Definations:
                    
                    1. Knowledge: Recalling facts, terms, basic concepts, or answers without necessarily understanding them
                    2. Comprehension: Demonstrating understanding of facts by interpreting, translating, summarizing, or explaining
                    3. Application: Using learned information in new concrete situations to solve problems
                    4. Analysis: Breaking down information into parts, examining relationships, distinguishing facts from inferences
                    5. Synthesis: Combining elements to form a new whole, proposing solutions, or designing new approaches
                    6. Evaluation: Making judgments based on criteria and standards through checking and critiquing

                    Now classify the following question:

                    Question: {query}
                    """,
            }
        ],
        model="openai/gpt-oss-120b",
    )

    reply = chat_completion.choices[0].message.content.lower()

    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        chat_completion = groq_client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": f"""Extract answer from previous reponse only in one word without punctuation from: 
                        [knowledge , comprehension , application , analysis, synthesis , evaluation]
                    previous response : {reply}""",
                }
            ],
            model="openai/gpt-oss-20b",
        )

        reply = chat_completion.choices[0].message.content.lower()

    ipo_pred_labels.append(reply.lower())

100%|██████████| 126/126 [05:03<00:00,  2.41s/it]


In [13]:
print(classification_report(label , [label_mapper[key.lower()] for key in ipo_pred_labels]))

              precision    recall  f1-score   support

           0       0.87      0.91      0.89        22
           1       0.64      0.70      0.67        20
           2       0.54      0.47      0.50        15
           3       0.86      0.63      0.73        19
           4       0.71      0.93      0.81        29
           5       1.00      0.76      0.86        21

    accuracy                           0.76       126
   macro avg       0.77      0.73      0.74       126
weighted avg       0.78      0.76      0.76       126



## LLAMA4-Scout

In [14]:
ipl_pred_labels = []

for query in tqdm(queries):
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""Given the query below, classify which label it belongs.
                    Labels: [knowledge, comprehension, application, analysis, synthesis, evaluation]

                    Definations:
                    
                    1. Knowledge: Recalling facts, terms, basic concepts, or answers without necessarily understanding them
                    2. Comprehension: Demonstrating understanding of facts by interpreting, translating, summarizing, or explaining
                    3. Application: Using learned information in new concrete situations to solve problems
                    4. Analysis: Breaking down information into parts, examining relationships, distinguishing facts from inferences
                    5. Synthesis: Combining elements to form a new whole, proposing solutions, or designing new approaches
                    6. Evaluation: Making judgments based on criteria and standards through checking and critiquing

                    Now classify the following question:

                    Question: {query}
                    """,
            }
        ],
        model="meta-llama/llama-4-scout-17b-16e-instruct",
    )

    reply = chat_completion.choices[0].message.content.lower()

    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        chat_completion = groq_client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": f"""Extract answer from previous reponse only in one word without punctuation from: 
                        [knowledge , comprehension , application , analysis, synthesis , evaluation]
                    previous response : {reply}""",
                }
            ],
            model="meta-llama/llama-4-scout-17b-16e-instruct",
        )

        reply = chat_completion.choices[0].message.content.lower()

    ipl_pred_labels.append(reply.lower())

100%|██████████| 126/126 [09:25<00:00,  4.48s/it]


In [15]:
print(classification_report(label , [label_mapper[key.lower()] for key in ipl_pred_labels]))

              precision    recall  f1-score   support

           0       0.87      0.91      0.89        22
           1       0.75      0.75      0.75        20
           2       0.33      0.27      0.30        15
           3       0.79      0.58      0.67        19
           4       0.68      0.90      0.78        29
           5       0.89      0.81      0.85        21

    accuracy                           0.74       126
   macro avg       0.72      0.70      0.70       126
weighted avg       0.74      0.74      0.73       126



# Chain-of-Thought without Context

## GPT-OSS-120B

In [17]:
coto_pred_labels = []

for query in tqdm(queries):
    # Reason

    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""
                    Given the query below, reason about which label it belongs.
                    Labels: [knowledge, comprehension, application, analysis, synthesis, evaluation]

                    Query: {query}""",
            }
        ],
        model="openai/gpt-oss-120b",
    )

    reply = chat_completion.choices[0].message.content.lower()

    # Summarize and Classify
    
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""
                    Summarize the reasoning into exactly one label from.
                    Labels: [knowledge, comprehension, application, analysis, synthesis, evaluation]

                    Reasoning: {reply}

                    Answer in one word ONLY""",
            }
        ],
        model="openai/gpt-oss-120b",
    )

    reply = chat_completion.choices[0].message.content.lower()

    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        chat_completion = groq_client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": f"""Extract answer from previous reponse only in one word without punctuation from:
                        [knowledge , comprehension , application , analysis, synthesis , evaluation]
                    previous response : {reply}""",
                }
            ],
            model="openai/gpt-oss-20b",
        )

        reply = chat_completion.choices[0].message.content.lower()
    time.sleep(10)

    coto_pred_labels.append(reply.lower())

100%|██████████| 126/126 [23:28<00:00, 11.18s/it]


In [18]:
print(classification_report(label , [label_mapper[key.lower()] for key in coto_pred_labels]))

              precision    recall  f1-score   support

           0       0.83      0.86      0.84        22
           1       0.81      0.65      0.72        20
           2       0.54      0.47      0.50        15
           3       0.72      0.68      0.70        19
           4       0.69      0.93      0.79        29
           5       0.94      0.76      0.84        21

    accuracy                           0.75       126
   macro avg       0.76      0.73      0.73       126
weighted avg       0.76      0.75      0.75       126



## LLAMA4-Scout

In [19]:
cotl_pred_labels = []

for query in tqdm(queries):
    # Reason

    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""
                    Given the query below, reason about which label it belongs.
                    Labels: [knowledge, comprehension, application, analysis, synthesis, evaluation]

                    Query: {query}""",
            }
        ],
        model="meta-llama/llama-4-scout-17b-16e-instruct",
    )

    reply = chat_completion.choices[0].message.content.lower()

    # Summarize and Classify
    
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""
                    Summarize the reasoning into exactly one label from.
                    Labels: [knowledge, comprehension, application, analysis, synthesis, evaluation]

                    Reasoning: {reply}

                    Answer in one word ONLY""",
            }
        ],
        model="meta-llama/llama-4-scout-17b-16e-instruct",
    )

    reply = chat_completion.choices[0].message.content.lower()

    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        chat_completion = groq_client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": f"""Extract answer from previous reponse only in one word without punctuation from:
                        [knowledge , comprehension , application , analysis, synthesis , evaluation]
                    previous response : {reply}""",
                }
            ],
            model="meta-llama/llama-4-scout-17b-16e-instruct",
        )

        reply = chat_completion.choices[0].message.content.lower()
    time.sleep(10)

    cotl_pred_labels.append(reply.lower())

100%|██████████| 126/126 [24:24<00:00, 11.62s/it]


In [20]:
print(classification_report(label , [label_mapper[key.lower()] for key in cotl_pred_labels]))

              precision    recall  f1-score   support

           0       0.95      0.86      0.90        22
           1       0.82      0.70      0.76        20
           2       0.27      0.20      0.23        15
           3       0.74      0.74      0.74        19
           4       0.63      0.90      0.74        29
           5       0.89      0.76      0.82        21

    accuracy                           0.73       126
   macro avg       0.72      0.69      0.70       126
weighted avg       0.73      0.73      0.72       126



# Save Labels

In [ ]:
label_data = {
    'zero_shot_no_context_gpt' : zso_pred_labels,
    'zero_shot_no_context_llama' : zsl_pred_labels,
    'instruct_prompt_no_context_gpt' : ipo_pred_labels, 
    'instruct_prompt_no_context_llama' : ipl_pred_labels
            }

df = pd.DataFrame(data= label_data)

df.to_csv('no_context.csv', index=False) 

In [ ]:
label_data = {
    'cot_no_context_gpt' : coto_pred_labels,
    'cot_no_context_llama' : cotl_pred_labels
            }

df = pd.DataFrame(data= label_data)

df.to_csv('no_context_cot.csv', index=False) 